# Atlas-5K 10M on WEATHER-5K: ParadoxAI Lab

The adopted 1M model (v2c) is **grown to 9.64M parameters** (same forecasts at the start) and trained on. Rules and
predictions were written in `atlas/README.md` ("Atlas-5K 10M") before this notebook ran.

**How to run:** Runtime → Change runtime type → **GPU** (A100 or L4 if offered, T4 works too) → Save. Then **Runtime → Run all**.
* Everything is saved in your Google Drive folder `PWM-10M` (dataset copy, checkpoints, logs, results).
* If the session ends or disconnects, open the notebook again and **Run all**: training resumes from the last
  checkpoint (at most 2,000 steps are lost). Keep this browser tab open while it runs.
* The 2023 test year is used only if the rule adopts the 10M model (cell 4), and once for v2c (cell 5).


In [ ]:
# 1. Google Drive (storage for data, checkpoints and results) and the GPU
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, json, glob
W = '/content/drive/MyDrive/PWM-10M'
for d in ('out', 'data'):
    os.makedirs(f'{W}/{d}', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
# 2. Code (public repository, nothing to sign in to)
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1

In [ ]:
# 3. Data: WEATHER-5K (5,672 stations, hourly 2014-2023). First time: download from the authors (6.4 GB) and convert
#    (about 15-25 min), then keep a 5 GB copy in Drive. Later sessions: copy from Drive (a few minutes).
R = '/content/raw'; os.makedirs(R, exist_ok=True)
X, M = f'{W}/data/W5K_X.npy', f'{W}/data/W5K_META.npz'
if os.path.exists(X) and os.path.getsize(X) > 4_900_000_000 and os.path.exists(M):
    print('copying WEATHER-5K from Drive ...'); shutil.copy(M, R); shutil.copy(X, R)
else:
    !cd /content/pwm/atlas/delhi/tools && python -u fetch_weather5k.py --out /content/raw --tmp /content/tmp --workers 8
    print('saving a copy to Drive ...'); shutil.copy(f'{R}/W5K_META.npz', M); shutil.copy(f'{R}/W5K_X.npy', X)
!ls -la /content/raw

In [ ]:
# 4. Train the 10M model (resumes automatically). The log prints s/step and the time left; check G0: the 'step 0'
#    validation should be about 0.2796 (v2c's best on the same stations).
%cd /content/pwm/atlas/delhi
os.environ['PW_RAW'] = '/content/raw'; os.environ['ATLAS_OUT'] = f'{W}/out'
TAG = 'w5k_atlas_10m_h24_g1'
!python -u run_atlas_5k.py --init ../../models/series/57_atlas-5k_v2c_weather5k_adopted.pt --grow 2,7,7 --k 32 --steps 80000 --lr 3e-4 --patience 8 --tag {TAG} --no-test --test-if-below 0.2786 --pause-at-cap --cap 660 2>&1 | grep --line-buffered -v "Warning\|warn(" | tee -a "{W}/out/log_{TAG}.txt" 

In [ ]:
# 5. v2c (the 1M entry), scored once on the 2023 test year as pre-registered. Skipped if already done.
if not os.path.exists(f'{W}/out/w5k_atlas_1m_h24_v2c_final_s600.json'):
    !python -u run_atlas_5k.py --eval ../../models/series/57_atlas-5k_v2c_weather5k_adopted.pt --k 32 --tag w5k_atlas_1m_h24_v2c_final 2>&1 | grep --line-buffered -v "Warning\|warn(" | tee -a "{W}/out/log_w5k_atlas_1m_h24_v2c_final.txt"
else:
    print('v2c test already done')

In [ ]:
# 6. Summary against the published leaders (WEATHER-5K, 24 h, MAE; lower is better)
PUB = {'PhysicsFormer (19.3M)': [1.55, 1.59, 57.37, 1.21, 1.43], 'ECMWF-HRES': [1.76, 1.85, 63.8, 1.48, 0.86]}
NAMES = ['Temperature', 'Dewpoint', 'Wind direction', 'Wind speed', 'Sea-level pressure']
for f in sorted(glob.glob(f'{W}/out/*.json')):
    r = json.load(open(f))
    print(os.path.basename(f), '| params', r.get('params'), '| GPU', r.get('gpu'), '| val E', r.get('val_E'), '| adopted', r.get('adopted'))
    if 'test' in r:
        t = [r['test']['MAE'][n] for n in NAMES]
        print('   test 2023 MAE  ', [round(x, 3) for x in t])
        for k, p in PUB.items():
            wins = [n for n, a, b in zip(NAMES, t, p) if a < b]
            print(f'   lower than {k} on {len(wins)}/5: {wins}')
print('Files in Drive:', sorted(os.listdir(f'{W}/out')))